In [ ]:
import arcpy
from arcpy import env
import os
import numpy as np
from arcgis import GIS
from arcgis.features import GeoAccessor
from arcgis.features import GeoSeriesAccessor
import pandas as pd

arcpy.env.overwriteOutput = True
arcpy.env.parallelProcessingFactor = "90%"

# show all columns
pd.options.display.max_columns = None

# pd.pivot_table(df, values='a', index='b', columns='c', aggfunc='sum', fill_value=0)
# pd.DataFrame.spatial.from_featureclass(???)  
# df.spatial.to_featureclass(location=???,sanitize_columns=False)  

# gsa = arcgis.features.GeoSeriesAccessor(df['SHAPE'])  
# df['AREA'] = gsa.area  # KNOW YOUR UNITS

In [ ]:
# fill NA values in Spatially enabled dataframes (ignores SHAPE column)
def fill_na_sedf(df_with_shape_column, fill_value=0):
    if 'SHAPE' in list(df_with_shape_column.columns):
        cols_to_fill = df_with_shape_column.columns.difference(['SHAPE'])
        df_with_shape_column[cols_to_fill] = df_with_shape_column[cols_to_fill].fillna(fill_value)
        return df_with_shape_column
    else:
        raise Exception("Dataframe does not include 'SHAPE' column")

In [ ]:
#=====================
# create output dirs
#=====================

outputs = ['.\\Outputs', "scratch.gdb", 'Affordability_Housing_Transportation_Costs.gdb']

if not os.path.exists(outputs[0]):
    os.makedirs(outputs[0])

gdb = os.path.join(outputs[0], outputs[1])
gdb2 = os.path.join(outputs[0], outputs[2])

if not arcpy.Exists(gdb):
    arcpy.CreateFileGDB_management(outputs[0], outputs[1])

if not arcpy.Exists(gdb2):
    arcpy.CreateFileGDB_management(outputs[0], outputs[2])

In [ ]:
#=====================
# load city area shapefile
#=====================
city_area_shp = pd.DataFrame.spatial.from_featureclass(r".\Inputs\city_area_with_workshop_areas.shp")
city_area_shp.head()

In [ ]:
#=====================
# load housing transportations costs data
#=====================

# Heads up: the columns from this table were reading in as object, not numeric
ht_df = pd.read_csv(r".\Inputs\H +T Costs for Dashboard 2019-2023 (GPI & TDM) - Composite H + T Metric.csv")
ht_df = ht_df.replace(-1,0)

ht_df.head()

In [ ]:
#=====================
# compare names in city area column to ensure same spelling
#=====================

# A
city_areas_from_csv = ht_df['City Area'].to_list()

# B
city_area_from_shape = city_area_shp['CITY_NAME'].to_list()

# Values in A but NOT in B
in_a_not_b = list(set(city_areas_from_csv) - set(city_area_from_shape)) # 
print('in city_areas_from_csv, not city_area_from_shape: ',in_a_not_b)

# Values in B but NOT in A
in_b_not_a = list(set(city_area_from_shape) - set(city_areas_from_csv)) 
print('in city_area_from_shape, not city_areas_from_csv: : ',in_b_not_a)

# # Values that are unique to either list (Symmetric Difference)
# unique_to_one = list(set(city_areas_from_csv) ^ set(city_area_from_shape)) 
# print('unique_to_one',unique_to_one)

In [ ]:
#=====================
# format the tables prior to merging
#=====================

# rename columns
ht_df.rename({'City Area':'CITYAREA'},inplace=True, axis=1)
ht_df.drop('County', inplace=True, axis=1)
# ht_df.rename({'County':'CO_NAME'},inplace=True, axis=1)
city_area_shp.rename({'CITY_NAME':'CITYAREA'},inplace=True, axis=1)
# ht_df['CO_NAME'] = ht_df['CO_NAME'].str.upper()

# merge on city area
merged_df = city_area_shp[['CITYAREA', 'SUBAREA', 'CO_NAME', 'SHAPE']].merge(ht_df, on='CITYAREA',how='left')
merged_df = fill_na_sedf(merged_df)


 # update names of workshop areas, if present
wa_lookup = {
    'Box Elder (WFRC)': 'Box Elder Wfrc',
    'North Davis County': 'Davis County North',
    'South Davis County': 'Davis County South',
    'North Salt Lake County': 'Salt Lake County North',
    'Southwest Salt Lake County': 'Salt Lake County Sw',
    'Southeast Salt Lake County': 'Salt Lake County Se',
    'North Weber County': 'Weber County North',
    'South Weber County': 'Weber County South',
    'Central Utah County': 'Utah County Central',
    'North Utah County': 'Utah County North',
    'South Utah County': 'Utah County South'
}
merged_df['SUBAREA'] = merged_df['SUBAREA'].replace(wa_lookup)
merged_df.head()

In [ ]:
#=====================
# export to feature class
#=====================

# might be issues updating this; misspelled transportation the first time around

# this gdb should be zipped and uploaded to AGOL
merged_df.spatial.to_featureclass(location=os.path.join(gdb2, 'Affordability_Housing_Transportation_Costs'),sanitize_columns=False) 